In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torch.utils.data import TensorDataset, Subset

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

In [ ]:
def _is_power_of_two(n: int) -> bool:
    return (n & (n - 1) == 0) and n > 0

@torch.no_grad()
def _qiskit_hadamard_unitary(n: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    assert _is_power_of_two(n), "Hadamard size has to be power of 2"
    m = int(math.log2(n))

    qc = QuantumCircuit(m)
    for q in range(m):
        qc.h(q)

    U = Operator(qc).data

    U = torch.tensor(U.real, device=device, dtype=dtype)
    return U

class HTPerceptron2D(nn.Module):
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int, 
            H_size: int, 
            pods: int = 3, 
            use_gain: bool = True, 
            use_threshold: bool = True, 
            bias_1x1: bool = False
        ):
        super().__init__()
        
        assert _is_power_of_two(H_size), "H_size must be a power of 2 (e.g., 32)"
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.H_size = H_size
        self.pods = pods
        self.use_gain = use_gain
        self.use_threshold = use_threshold

        self.V = nn.ModuleList([nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias_1x1) for _ in range(pods)])

        # create nn module to use gain
        if use_gain:
            self.A = nn.ParameterList([nn.Parameter(torch.rand(1, 1, H_size, H_size)) for _ in range(pods)])
        else:
            self.register_buffer("_A_ones", torch.ones(1, 1, H_size, H_size))
            self.A = None

        # create nn module to use soft threshold
        if use_threshold:
            self.T = nn.ParameterList([nn.Parameter(0.1 * torch.rand(1, 1, H_size, H_size)) for _ in range(pods)])
        else:
            self.register_buffer("_T_zeros", torch.zeros(1, 1, H_size, H_size))
            self.T = None

        self.register_buffer("_H", None, persistent=False)
        self._H_cached_device = None
        self._H_cached_dtype = None

    def _ensure_H(self, device: torch.device, dtype: torch.dtype):
        if (self._H is None) or (self._H_cached_device != device) or (self._H_cached_dtype != dtype):
            H = _qiskit_hadamard_unitary(self.H_size, device=device, dtype=dtype)
            self._H = H
            self._H_cached_device = device
            self._H_cached_dtype = dtype

    def _ht_1d_last(self, u: torch.Tensor) -> torch.Tensor:
        return u @ self._H

    def _ht2d(self, x: torch.Tensor) -> torch.Tensor:
        x = self._ht_1d_last(x)

        x = x.transpose(-2, -1)
        x = self._ht_1d_last(x)
        x = x.transpose(-2, -1)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, Cin, H, W = x.shape
        if Cin != self.in_channels:
            raise ValueError(f"Expected in_channels={self.in_channels}, got {Cin}")

        self._ensure_H(device=x.device, dtype=x.dtype)

        if H > self.H_size or W > self.H_size:
            raise ValueError(f"Input spatial {(H, W)} exceeds H_size={self.H_size}")

        pad_h = self.H_size - H
        pad_w = self.H_size - W
        if pad_h or pad_w:
            x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        else:
            x_pad = x

        X = self._ht2d(x_pad)

        Ys = []
        for i in range(self.pods):
            Ai = self.A[i] if self.use_gain else self._A_ones
            Ti = self.T[i] if self.use_threshold else self._T_zeros

            Xi = X * Ai
            Zi = self.V[i](Xi)

            Yi = torch.sign(Zi) * F.relu(torch.abs(Zi) - Ti)

            Ys.append(Yi)

        Y = torch.stack(Ys, dim=0).sum(dim=0)

        y_pad = self._ht2d(Y)

        y = y_pad[..., :H, :W]
        return y


In [3]:
# download format
# turns MNIST images to PyTorch tensors and normalizes between [-1,1] centered at 0
train_transform = transforms.Compose([
    transforms.RandomCrop(28, padding=1),
    #transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# download data to computer
raw_train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

train_eval_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=test_transform
)

raw_test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,            # testing data so labels are unknown during training
    download=True,
    transform=test_transform
)

In [4]:
# use raw_train_small and raw_test_small to decrease number of training and test images
# use raw_train_dataset and raw_test_dataset for entire dataset
raw_train_subset = Subset(raw_train_dataset, range(10000))
raw_test_subset = Subset(raw_test_dataset, range(1000))

train_dataset = raw_train_dataset
test_dataset  = raw_test_dataset

In [5]:
# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=256,         # each epoch is 128 samples
    shuffle=True,            # randomize after each training epoch
    num_workers=4,
    pin_memory=True
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,         # each epoch is 256 samples
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# test shapes of pytorch datasets
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([256, 1, 28, 28])
torch.Size([256])


In [6]:
# CNN model
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.ht = HTPerceptron2D(in_channels=1, out_channels=32, H_size=32, use_gain=True)

        self.conv1a = nn.Conv2d(32, 32, 3, padding=1)
        self.conv1b = nn.Conv2d(32, 32, 3, padding=1)

        self.conv2a = nn.Conv2d(32, 64, 3, padding=1)
        self.conv2b = nn.Conv2d(64, 64, 3, padding=1)

        self.conv3a = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3b = nn.Conv2d(128, 128, 3, padding=1)

        self.conv4a = nn.Conv2d(128, 256, 3, padding=1)
        self.conv4b = nn.Conv2d(256, 256, 3, padding=1)

        # a and b batch norms for their corresponding conv layer because batchnorm learns
        self.bn0 = nn.BatchNorm2d(32)

        self.bn1a = nn.BatchNorm2d(32)
        self.bn2a = nn.BatchNorm2d(64)
        self.bn3a = nn.BatchNorm2d(128)
        self.bn4a = nn.BatchNorm2d(256)

        self.bn1b = nn.BatchNorm2d(32)
        self.bn2b = nn.BatchNorm2d(64)
        self.bn3b = nn.BatchNorm2d(128)
        self.bn4b = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        self.drop = nn.Dropout(0.2)

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)
        #self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.ht(x)
        x = F.relu(self.bn0(x))

        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))

        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))

        x = F.relu(self.bn3a(self.conv3a(x)))
        x = self.pool(F.relu(self.bn3b(self.conv3b(x))))

        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))

        #x = torch.flatten(x, start_dim=1)

        x = self.gap(x).squeeze(-1).squeeze(-1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)

        #x = self.drop(x)
        #x = self.fc3(x)

        return x

# create model, loss, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

num_epochs = 75 # NUM EPOCHS

model = CNN().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

cuda


In [7]:
# eval accuracy within training loop
def eval_acc_loader(loader):
    was_training = model.training
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            pred = model(images).argmax(1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    if was_training:
        model.train()
    return 100.0 * correct / total

# eval loss within training loop
def eval_loss_loader(loader):
    was_training = model.training
    model.eval()
    total_loss = 0.0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)
            bs = labels.size(0)
            total_loss += loss.item() * bs
            total += bs
    if was_training:
        model.train()
    return total_loss / total

In [8]:
# training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    train_acc = eval_acc_loader(train_eval_loader)
    test_acc  = eval_acc_loader(test_loader)

    train_loss_eval = eval_loss_loader(train_eval_loader)
    test_loss_eval  = eval_loss_loader(test_loader)

    print("lr:", optimizer.param_groups[0]["lr"])
    print(f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
    print(f"Train Loss(eval): {train_loss_eval:.4f} | Test Loss(eval): {test_loss_eval:.4f}")
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    print()

lr: 0.0004997807075247146
Train Acc: 87.14% | Test Acc: 86.09%
Train Loss(eval): 0.4719 | Test Loss(eval): 0.4945
Epoch [1/75], Loss: 0.6843

lr: 0.0004991232148123761
Train Acc: 88.13% | Test Acc: 86.99%
Train Loss(eval): 0.4413 | Test Loss(eval): 0.4721
Epoch [2/75], Loss: 0.4735

lr: 0.0004980286753286195
Train Acc: 89.85% | Test Acc: 88.64%
Train Loss(eval): 0.3950 | Test Loss(eval): 0.4265
Epoch [3/75], Loss: 0.4265

lr: 0.0004964990092676262
Train Acc: 89.35% | Test Acc: 88.02%
Train Loss(eval): 0.4107 | Test Loss(eval): 0.4516
Epoch [4/75], Loss: 0.4011

lr: 0.0004945369001834514
Train Acc: 90.41% | Test Acc: 88.82%
Train Loss(eval): 0.3716 | Test Loss(eval): 0.4167
Epoch [5/75], Loss: 0.3794

lr: 0.0004921457902821577
Train Acc: 91.88% | Test Acc: 90.01%
Train Loss(eval): 0.3379 | Test Loss(eval): 0.3825
Epoch [6/75], Loss: 0.3643

lr: 0.0004893298743830167
Train Acc: 92.77% | Test Acc: 90.70%
Train Loss(eval): 0.3155 | Test Loss(eval): 0.3639
Epoch [7/75], Loss: 0.3518

lr: 0.